In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import sample_colorscale
from plotly.subplots import make_subplots

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
RESULT_CSV = None  # e.g. PLL_DIR / "pll_test_vctrl_S5_20261021_100000.csv"; None = newest run

if RESULT_CSV is None:
    RESULT_CSV = sorted((p for p in PLL_DIR.glob("pll_test_vctrl_*.csv") if not p.stem.endswith("_traces")),
                        key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
TRACE_CSV = RESULT_CSV.with_name(RESULT_CSV.stem + "_traces.csv")
all_runs = pd.read_csv(RESULT_CSV).sort_values("vctrl_pred").reset_index(drop=True)
traces = pd.read_csv(TRACE_CSV) if TRACE_CSV.exists() else pd.DataFrame(columns=["vctrl_target", "kind"])

# Vctrl on the x axes: measured on the scope when there is a reading, else the table's prediction
HAS_SCOPE = all_runs["vctrl_meas"].notna().any()
X_COL = "vctrl_meas" if HAS_SCOPE else "vctrl_pred"
X_TITLE = "Vctrl measured on the scope [V]" if HAS_SCOPE else "Vctrl predicted by the table [V] (no scope)"

runs = all_runs[(all_runs["locked"].fillna(0) == 1) & (all_runs["status"] == "ok")].reset_index(drop=True)
print(f"{RESULT_CSV.name}: {len(all_runs)} settings at {all_runs['f_vco'].iloc[0] / 1e6:g} MHz, "
      + (f"Vctrl pad {all_runs['vctrl_pad'].iloc[0]} (read: on to read Vctrl, off for the measurement), "
         if "vctrl_pad" in all_runs else f"set_v_ctrl {all_runs['set_v_ctrl'].iloc[0]:.0f} (2: Vctrl pad on), ")
      + f"{(all_runs['locked'].fillna(0) == 1).sum()} locked, {len(runs)} measured ok; "
      + (f"probe on scope CH{all_runs['scope_ch'].iloc[0]:.0f} ({all_runs['probe_att'].iloc[0]:g}:1)" if HAS_SCOPE
         else "no scope (probe removed)"))

pd.DataFrame({
    "setting": [f"c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} max {r.vco_current_max:.0f}"
                for r in all_runs.itertuples()],
    "Vctrl target [V]": all_runs["vctrl_target"],
    "Vctrl pred [V]": all_runs["vctrl_pred"],
    "Vctrl meas [V]": all_runs["vctrl_meas"],
    "Vctrl end [V]": all_runs["vctrl_meas_end"],
    "ripple pk-pk [mV]": all_runs["vctrl_pkpk"] * 1e3,
    "ripple RMS [mV]": all_runs["vctrl_ac_rms"] * 1e3,
    "|Kvco| pred [MHz/V]": all_runs["kvco_pred"] / 1e6,
    "locked": all_runs["locked"],
    "carrier [dBm]": all_runs["carrier_dbm"],
    "jitter curve [ps]": all_runs["time_jitter_trace_ps"],
    "jitter noise [ps]": all_runs["time_jitter_noise_ps"],
    "jitter spurs [ps]": all_runs["time_jitter_spur_ps"],
    "status": all_runs["status"],
}).round(3)

pll_test_vctrl_S5_20261020_131332.csv: 8 settings at 1000 MHz, set_v_ctrl 2 (2: Vctrl pad on), 6 locked, 5 measured ok; probe on scope CH1 (10:1)


,setting,Vctrl target [V],Vctrl pred [V],Vctrl meas [V],Vctrl end [V],ripple pk-pk [mV],ripple RMS [mV],|Kvco| pred [MHz/V],locked,carrier [dBm],jitter curve [ps],jitter noise [ps],jitter spurs [ps],status
0,c9 min 13 max 12,0.472,0.476,NaN,NaN,NaN,NaN,953.615,0,NaN,NaN,NaN,NaN,not_locked
1,c10 min 14 max 0,0.501,0.500,NaN,NaN,NaN,NaN,2005.039,0,NaN,NaN,NaN,NaN,not_locked
2,c9 min 11 max 8,0.531,0.541,0.492,0.492,160.622,40.304,1964.530,1,1.154,773.607,98.268,767.341,ok
3,c3 min 11 max 15,0.561,0.561,0.524,0.524,194.138,41.642,1851.528,1,-6.277,4712.263,1887.598,4317.684,ok
4,c6 min 9 max 12,0.590,0.600,0.586,0.586,168.560,40.736,2102.223,1,-1.293,2357.669,470.372,2310.271,ok
5,c6 min 8 max 10,0.620,0.613,0.597,0.598,171.157,40.743,1995.819,1,-0.390,1849.001,575.995,1756.996,overload
6,c2 min 11 max 14,0.649,0.649,0.628,0.630,195.118,42.557,2475.655,1,-8.880,5776.892,3404.751,4666.921,ok
7,c2 min 11 max 0,0.679,0.679,0.657,0.657,192.717,41.073,1472.959,1,-3.760,3920.694,2261.927,3202.426,ok


In [3]:
import sys

sys.path.insert(0, str(PLL_DIR.parents[1]))  # repo root
from sw.lib.pll_util import bin_phase_noise

INK, GRID = "#52514e", "#e4e3df"
OFFSET_AXIS = dict(type="log", tickvals=[1e2, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8],
                   ticktext=["100 Hz", "1 kHz", "10 kHz", "100 kHz", "1 MHz", "10 MHz", "100 MHz"],
                   minor=dict(showgrid=True, gridcolor="#f1f0ed"))
N_BINS = 300  # points per plotted phase-noise curve; None = every measured point
L_OFFSETS = (1e3, 1e4, 1e5, 1e6)  # [Hz] spot levels read from the full curve (power mean within +-10 %)
SPUR_F = 1.95e6  # [Hz] the interferer seen in every run so far


def vctrl_color(v):
    """Colour of a Vctrl [V] on one scale for all plots (low: blue, high: yellow)."""
    lo, hi = all_runs[X_COL].min(), all_runs[X_COL].max()
    return sample_colorscale("Viridis", [0.0 if hi == lo else (v - lo) / (hi - lo)])[0]


def trace_of(vctrl_target, kind):
    return traces[(traces["vctrl_target"] == vctrl_target) & (traces["kind"] == kind)]


def curve_level(curve, f):
    """Level [dBc/Hz] of the curve (columns freq, level_dbm) at offset f: power mean of the points within +-10 %."""
    near = curve[(curve["freq"] >= 0.9 * f) & (curve["freq"] <= 1.1 * f)]
    if near.empty:
        return np.nan
    return 10 * np.log10(np.mean(10 ** (near["level_dbm"] / 10)))


def spur_level(spurs, f, tol=0.02):
    """Level [dBc on the pad] of the spur closest to f [Hz] (within tol relative) in a spurs string
    ("<f> Hz: <level> dBc; ..."); NaN when there is none."""
    best = np.nan
    for part in str(spurs).split(";"):
        if "Hz:" not in part:
            continue
        f_txt, level_txt = part.split("Hz:")
        if abs(float(f_txt) - f) <= tol * f:
            best = np.nanmax([best, float(level_txt.replace("dBc", ""))])
    return best


def binned(df, f_start, f_stop):
    """The curve in df (freq, level_dbm) between f_start and f_stop averaged into N_BINS log-spaced bins in linear
    power (pll_util.bin_phase_noise): display only, the jitter uses the measured points."""
    df = df[(df["freq"] >= f_start) & (df["freq"] <= f_stop)].sort_values("freq")
    if N_BINS is None or len(df) < 2:
        return df
    freq, level = bin_phase_noise(df["freq"], df["level_dbm"], np.geomspace(f_start, f_stop, N_BINS + 1))
    return pd.DataFrame({"freq": freq, "level_dbm": level})


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to the CSV, then show the figure."""
    base = RESULT_CSV.with_name(f"{RESULT_CSV.stem}_{name}")
    fig.write_html(base.with_suffix(".html"))
    try:
        fig.write_image(base.with_suffix(".png"), width=1100, height=fig.layout.height or 600, scale=2)
        print(f"saved {base}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {base}.html; PNG export failed: {e}")
    fig.show()


# Performance metrics per measured setting (VCO-referred levels: pad + 20 log10 N)
for f in L_OFFSETS:
    runs[f"L_vco_{f:.0e}"] = [curve_level(trace_of(r.vctrl_target, "L_trace"), f) + 20 * np.log10(r.total_div)
                              for r in runs.itertuples()]
runs["spur_1p95_dbc"] = [spur_level(s, SPUR_F) for s in runs["spurs"]]

In [4]:
# Is the scope reading the operating point the table predicts, and how much ripple sits on the control node?
# Left: measured against predicted Vctrl (on the diagonal: the table is right; after the analyzer measurement as an
# open circle: drift). Right: ripple (peak-to-peak and AC RMS) against Vctrl; unlocked settings as x.
if not HAS_SCOPE:
    print("no scope readings in this run")
else:
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                        subplot_titles=("measured vs predicted Vctrl", "ripple on the control node"))
    locked = all_runs["locked"].fillna(0) == 1
    lo, hi = all_runs[["vctrl_pred", "vctrl_meas"]].min().min(), all_runs[["vctrl_pred", "vctrl_meas"]].max().max()
    fig.add_scatter(x=[lo, hi], y=[lo, hi], mode="lines", line=dict(color=GRID, dash="dot"), showlegend=False,
                    row=1, col=1)
    for mask, symbol, name in ((locked, "circle", "locked"), (~locked, "x", "not locked")):
        d = all_runs[mask]
        fig.add_scatter(x=d["vctrl_pred"], y=d["vctrl_meas"], mode="markers", name=name,
                        marker=dict(symbol=symbol, size=10, color=INK), row=1, col=1,
                        hovertemplate="pred %{x:.3f} V, meas %{y:.3f} V<extra></extra>")
        fig.add_scatter(x=d["vctrl_meas"], y=d["vctrl_pkpk"] * 1e3, mode="markers", name="pk-pk", showlegend=False,
                        marker=dict(symbol=symbol, size=10, color="#2a78d6"), row=1, col=2,
                        hovertemplate="%{x:.3f} V: %{y:.1f} mV pk-pk<extra></extra>")
        fig.add_scatter(x=d["vctrl_meas"], y=d["vctrl_ac_rms"] * 1e3, mode="markers", name="RMS", showlegend=False,
                        marker=dict(symbol=symbol, size=10, color="#eb6834"), row=1, col=2,
                        hovertemplate="%{x:.3f} V: %{y:.2f} mV RMS<extra></extra>")
    fig.add_scatter(x=all_runs["vctrl_pred"], y=all_runs["vctrl_meas_end"], mode="markers", name="after measurement",
                    marker=dict(symbol="circle-open", size=12, color=INK), row=1, col=1)
    fig.update_layout(title="Control voltage on the scope (blue: peak-to-peak, orange: AC RMS)",
                      template="plotly_white", height=500)
    fig.update_xaxes(gridcolor=GRID)
    fig.update_yaxes(gridcolor=GRID)
    fig.update_xaxes(title="Vctrl predicted [V]", row=1, col=1)
    fig.update_yaxes(title="Vctrl measured [V]", row=1, col=1)
    fig.update_xaxes(title="Vctrl measured [V]", row=1, col=2)
    fig.update_yaxes(title="ripple [mV]", rangemode="tozero", row=1, col=2)
    save(fig, "vctrl_scope")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_test_vctrl_S5_20261020_131332_vctrl_scope.html and .png


In [5]:
# Performance against Vctrl. Top left: RMS jitter on the pad (full curve, its noise and spur parts). Top right: the
# full curve at spot offsets, referred to the VCO. Bottom left: the ~1.95 MHz interferer (dBc on the pad). Bottom
# right: |Kvco| of each setting (the other thing that changes with the VCO codes: if it follows Vctrl, the two cannot
# be told apart from this run alone). The table below: rank correlation of every metric with Vctrl and with |Kvco|.
METRICS = {"time_jitter_trace_ps": "jitter curve [ps]", "time_jitter_noise_ps": "jitter noise [ps]",
           "time_jitter_spur_ps": "jitter spurs [ps]", "spur_1p95_dbc": "1.95 MHz spur [dBc]",
           "carrier_dbm": "carrier [dBm]", "vctrl_pkpk": "ripple pk-pk [V]", "vctrl_ac_rms": "ripple RMS [V]",
           **{f"L_vco_{f:.0e}": f"L({f:.0e} Hz) at the VCO [dBc/Hz]" for f in L_OFFSETS}}

fig = make_subplots(rows=2, cols=2, horizontal_spacing=0.1, vertical_spacing=0.14,
                    subplot_titles=("RMS jitter on the pad", "phase noise at the VCO", "~1.95 MHz interferer",
                                    "|Kvco| of the setting"))
for col, color, name in (("time_jitter_trace_ps", INK, "full curve"), ("time_jitter_noise_ps", "#2a78d6", "noise"),
                         ("time_jitter_spur_ps", "#eb6834", "spurs")):
    fig.add_scatter(x=runs[X_COL], y=runs[col], mode="lines+markers", name=name, line=dict(color=color),
                    row=1, col=1, hovertemplate="%{x:.3f} V: %{y:.1f} ps<extra>" + name + "</extra>")
for f, color in zip(L_OFFSETS, ("#1baf7a", "#eda100", "#e87ba4", "#4a3aa7")):
    name = f"L({f / 1e3:g} kHz)"
    fig.add_scatter(x=runs[X_COL], y=runs[f"L_vco_{f:.0e}"], mode="lines+markers", name=name, line=dict(color=color),
                    row=1, col=2, hovertemplate="%{x:.3f} V: %{y:.1f} dBc/Hz<extra>" + name + "</extra>")
fig.add_scatter(x=runs[X_COL], y=runs["spur_1p95_dbc"], mode="lines+markers", name="1.95 MHz spur",
                line=dict(color="#e34948"), row=2, col=1, hovertemplate="%{x:.3f} V: %{y:.1f} dBc<extra></extra>")
fig.add_scatter(x=all_runs[X_COL], y=all_runs["kvco_pred"] / 1e6, mode="markers", name="|Kvco| (all settings)",
                marker=dict(size=9, color=INK, symbol=["circle" if l == 1 else "x" for l in all_runs["locked"]]),
                row=2, col=2, hovertemplate="%{x:.3f} V: %{y:.0f} MHz/V<extra></extra>")
fig.update_layout(title=f"Performance against the control voltage, {all_runs['f_vco'].iloc[0] / 1e6:g} MHz "
                        f"(x: {X_COL})", template="plotly_white", height=750, legend=dict(orientation="h", y=-0.12))
fig.update_xaxes(title=X_TITLE, gridcolor=GRID)
fig.update_yaxes(gridcolor=GRID)
fig.update_yaxes(title="RMS jitter [ps]", type="log", row=1, col=1)
fig.update_yaxes(title="L(f) at the VCO [dBc/Hz]", row=1, col=2)
fig.update_yaxes(title="spur on the pad [dBc]", row=2, col=1)
fig.update_yaxes(title="|Kvco| [MHz/V]", row=2, col=2)
save(fig, "metrics_vs_vctrl")

# Spearman rank correlation (monotonic relation, -1..1) with Vctrl and with |Kvco|; few points: read it as a hint
cols = [c for c in METRICS if c in runs and runs[c].notna().sum() >= 3]
corr = runs[cols + ["vctrl_pred", "vctrl_meas", "kvco_pred"]].corr(method="spearman")
pd.DataFrame({"metric": [METRICS[c] for c in cols],
              "vs Vctrl measured": corr.loc[cols, "vctrl_meas"].values,
              "vs Vctrl predicted": corr.loc[cols, "vctrl_pred"].values,
              "vs |Kvco|": corr.loc[cols, "kvco_pred"].values}).round(2)

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_test_vctrl_S5_20261020_131332_metrics_vs_vctrl.html and .png


,metric,vs Vctrl measured,vs Vctrl predicted,vs |Kvco|
0,jitter curve [ps],0.5,0.5,0.2
1,jitter noise [ps],0.8,0.8,0.1
2,jitter spurs [ps],0.5,0.5,0.2
3,carrier [dBm],-0.5,-0.5,-0.2
4,ripple pk-pk [V],0.5,0.5,0.2
5,ripple RMS [V],0.5,0.5,0.2
6,L(1e+03 Hz) at the VCO [dBc/Hz],0.8,0.8,0.1
7,L(1e+04 Hz) at the VCO [dBc/Hz],0.8,0.8,0.1
8,L(1e+05 Hz) at the VCO [dBc/Hz],0.8,0.8,0.1
9,L(1e+06 Hz) at the VCO [dBc/Hz],0.6,0.6,0.5


In [6]:
F_start, F_stop = 1e3, 20e6  # offset range shown [Hz]

# Full phase-noise curve per setting, referred to the VCO (pad + 20 log10 N), coloured by Vctrl (blue low, yellow
# high): shows where in offset the curves split (in-band floor, loop peak, interferers).
fig = go.Figure()
for r in runs.sort_values(X_COL).itertuples():
    lt = binned(trace_of(r.vctrl_target, "L_trace"), F_start, F_stop)
    if lt.empty:
        continue
    name = (f"Vctrl {getattr(r, X_COL):.3f} V, |Kvco| {r.kvco_pred / 1e6:.0f} MHz/V (c{r.vco_tune_coarse:.0f} "
            f"min {r.vco_current_min:.0f} max {r.vco_current_max:.0f}): {r.time_jitter_trace_ps:.0f} ps")
    fig.add_scatter(x=lt["freq"], y=lt["level_dbm"] + 20 * np.log10(r.total_div), mode="lines", name=name,
                    line=dict(color=vctrl_color(getattr(r, X_COL)), width=2),
                    hovertemplate="%{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
fig.update_layout(title=f"Phase noise referred to the VCO per Vctrl, {all_runs['f_vco'].iloc[0] / 1e6:g} MHz "
                        f"(reference {all_runs['f_ref'].iloc[0] / 1e6:.4g} MHz)",
                  template="plotly_white", height=600, legend=dict(orientation="h", y=-0.18),
                  hoverlabel=dict(namelength=-1))
fig.update_xaxes(title="offset from the carrier", gridcolor=GRID, **OFFSET_AXIS,
                 range=[np.log10(F_start), np.log10(F_stop)])
fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
save(fig, "phase_noise")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_test_vctrl_S5_20261020_131332_phase_noise.html and .png


In [7]:
F_VCO_TOL = 0.02  # relative: earlier runs within 2 % of this run's f_vco count as the same frequency

# This run against every earlier locked measurement in results/pll (pll_spectrum_kvco/_freq/_div, pll_range, other
# pll_test_vctrl runs) with a full-curve noise jitter: random jitter against the predicted Vctrl (the only Vctrl the
# runs without scope have). Filled: pad 20-100 MHz, open: pad outside it (older runs, less trusted). Big markers:
# same f_vco as this run. Same VCO codes at the same f_vco in an earlier run without probe (table below): shows
# whether the probe on the control node changes the loop.
history = []
for p in sorted(PLL_DIR.glob("pll_*.csv")):
    if p.stem.endswith("_traces") or p == RESULT_CSV:
        continue
    d = pd.read_csv(p)
    need = {"vctrl_pred", "time_jitter_noise_ps", "locked", "status", "f_vco", "f_out_meas"}
    if not need <= set(d.columns):
        continue  # marker-era or other result files
    d = d[(d["locked"].fillna(0) == 1) & (d["status"] == "ok")]
    if "set_v_ctrl" in d:
        d = d[d["set_v_ctrl"].fillna(0) != 2]  # Vctrl pad on: the pad disturbs the loop (2026-10-20), no reference
    history.append(d.assign(source=p.stem.rsplit("_", 3)[0], file=p.name))
history = pd.concat(history, ignore_index=True)  # at least the earlier pll_spectrum runs
same_f = (history["f_vco"] - all_runs["f_vco"].iloc[0]).abs() <= F_VCO_TOL * all_runs["f_vco"].iloc[0]
in_band = history["f_out_meas"].between(20e6, 100e6)

fig = go.Figure()
for i, (source, d) in enumerate(history.groupby("source")):
    color = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"][i % 6]
    fig.add_scatter(x=d["vctrl_pred"], y=d["time_jitter_noise_ps"], mode="markers", name=source,
                    marker=dict(color=color, size=np.where(same_f[d.index], 12, 7),
                                symbol=np.where(in_band[d.index], "circle", "circle-open")),
                    customdata=list(zip(d["f_vco"] / 1e6, d["file"])),
                    hovertemplate="%{customdata[0]:.0f} MHz, Vctrl %{x:.3f} V: %{y:.1f} ps<br>%{customdata[1]}"
                                  "<extra></extra>")
fig.add_scatter(x=runs["vctrl_pred"], y=runs["time_jitter_noise_ps"], mode="markers", name="this run",
                marker=dict(color=INK, size=14, symbol="star"),
                hovertemplate="Vctrl %{x:.3f} V: %{y:.1f} ps<extra>this run</extra>")
fig.update_layout(title="Random (noise) jitter on the pad against the predicted Vctrl, all runs",
                  template="plotly_white", height=550, legend=dict(orientation="h", y=-0.15))
fig.update_xaxes(title="Vctrl predicted by the table [V]", gridcolor=GRID)
fig.update_yaxes(title="noise jitter [ps]", type="log", gridcolor=GRID)
save(fig, "history_vs_vctrl")

codes = ["vco_tune_coarse", "vco_current_min", "vco_current_max", "set_current", "set_c1", "set_c2", "set_r1"]
codes = [c for c in codes if c in history]
now = runs[codes + ["vctrl_meas", "time_jitter_noise_ps", "time_jitter_trace_ps"]].rename(
    columns={"vctrl_meas": "Vctrl meas now [V]", "time_jitter_noise_ps": "noise now [ps]",
             "time_jitter_trace_ps": "curve now [ps]"})
before = history.loc[same_f, ["file", "vctrl_pred", "time_jitter_noise_ps", "time_jitter_trace_ps"] + codes].rename(
    columns={"vctrl_pred": "Vctrl pred [V]", "time_jitter_noise_ps": "noise before [ps]",
             "time_jitter_trace_ps": "curve before [ps]"})
before.merge(now, on=codes).round(3)

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_test_vctrl_S5_20261020_131332_history_vs_vctrl.html and .png


,file,Vctrl pred [V],noise before [ps],curve before [ps],vco_tune_coarse,vco_current_min,vco_current_max,set_current,set_c1,set_c2,set_r1,Vctrl meas now [V],noise now [ps],curve now [ps]
0,pll_test_vctrl_S5_20261020_130735.csv,0.541,33.032,68.182,9.0,11.0,8.0,7.0,0.0,0.0,7.0,0.492,98.268,773.607
1,pll_test_vctrl_S5_20261020_131031.csv,0.541,93.080,837.103,9.0,11.0,8.0,7.0,0.0,0.0,7.0,0.492,98.268,773.607
2,pll_test_vctrl_S5_20261020_131031.csv,0.561,1536.456,4214.321,3.0,11.0,15.0,7.0,0.0,0.0,7.0,0.524,1887.598,4712.263
